# arcroad — Phase 2a: Detour Index (v0.8, thin)

**This notebook computes nothing itself.** Every step it shows is a call into
`pipeline/`, which is versioned, tested on every push and tied to a commit hash.
It supersedes `HMA_Phase2a_DetourIndex_v07.ipynb`, whose pairing, graph, routing
and record cells now live in `pipeline/pairing.py`.

**What changed, and why.**

- **B-09.** v1.0 found two nodes on opposing carriageways, routed between them,
  then kept only the first. No value in `di_v10_points_2026-10-25.csv` can be
  re-routed. Every record now carries both endpoints (`a_*`, `b_*`), their OSM
  node ids, the extract date and the commit hash.
- **B-10.** v1.0 declared the HMDA box and did not clip to it: 1,549 points up to
  85 km out. The box is now a parameter (default: the declared HMDA box), the
  clip is applied, the excluded pairs are counted, and the box used is written
  into every record.
- **The export refuses invalid records.** `write_point_file` runs
  `provenance.validate_export` and writes nothing if a single record fails. A run
  that cannot pass validation does not get to produce a published figure.

**Comparability with v1.0.** The pairing algorithm is a literal port, checked
against the notebook cells on synthetic networks, and the default configuration
is v1.0's (`pipeline.pairing.V10`). Clipping happens after pairing, so inside the
box this run samples exactly what v1.0 sampled. On the same extract, expect
12,645 pairs before the clip, 1,549 outside, 11,096 kept. A different count means
the extract changed; say so before comparing medians.

Every figure here is a **candidate**. Nothing is a finding until a modelled route
has been checked against a driven one.

## 1. Install, and fetch the code this run will be tied to

In [ ]:
!pip -q install osmium tabulate
!apt-get -qq install -y osmium-tool > /dev/null 2>&1

REPO_URL = "https://github.com/rrmlogs8896/arcroad.git"
REPO_REF = "main"          # pin a commit hash here to reproduce an earlier run
REPO_DIR = "/content/arcroad"

import os, subprocess, sys
if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", "-q", REPO_URL, REPO_DIR], check=True)
subprocess.run(["git", "-C", REPO_DIR, "fetch", "-q", "origin"], check=True)
subprocess.run(["git", "-C", REPO_DIR, "checkout", "-q", REPO_REF], check=True)
subprocess.run(["git", "-C", REPO_DIR, "pull", "-q", "--ff-only"], check=False)
sys.path.insert(0, REPO_DIR)

from pipeline import pairing, metrics, provenance
SHA = pairing.commit_sha(REPO_DIR)
print("pipeline at", SHA)

## 2. Configuration, and Drive access up front

The study box and the run configuration are explicit and are recorded in the output. Drive is mounted here, at the start, and the destination folder must already exist (B-13).

In [ ]:
EXTRACT_URL = "https://download.geofabrik.de/asia/india/southern-zone-latest.osm.pbf"
PBF_PATH    = "/content/southern-zone.osm.pbf"
CLIP_PATH   = "/content/hyderabad_buffered.osm.pbf"
OUT_DIR     = "/content/arcroad_out"

STUDY_BBOX      = pairing.HMDA_BBOX   # B-10: the box the project declares
CONFIG          = pairing.V10         # v1.0's settings, so the result is comparable
CLIP_BUFFER_DEG = 0.20                # F-3: graph clipped wider than we measure

# B-13: where results land in Drive. Set once, here, and checked BEFORE the
# 20-minute run, so a wrong path or an unapproved Drive popup fails in seconds.
# The Anthropic folder lives inside AI PROJECTS — MASTER WORKSPACE since 25 Oct.
DRIVE_BASE = "/content/drive/MyDrive/AI PROJECTS — MASTER WORKSPACE/Anthropic/Hyderabad Mobility Atlas"
DRIVE_DEST = f"{DRIVE_BASE}/phase2a_detour_v08"

from google.colab import drive
drive.mount("/content/drive")            # approve the popup now, while you are here
if not os.path.isdir(DRIVE_BASE):
    raise SystemExit(f"Drive folder not found: {DRIVE_BASE}\n"
                     "Fix DRIVE_BASE rather than letting the run create a duplicate folder.")

os.makedirs(OUT_DIR, exist_ok=True)
print("study box", STUDY_BBOX, "| declared HMDA:", tuple(STUDY_BBOX) == pairing.HMDA_BBOX)
print("config", CONFIG)

## 3. Download and clip with a buffer (F-3)

The graph extends 0.2 degrees past the study box so routes near its edge are not cut short. Only the measured points are clipped to the box.

In [ ]:
if not (os.path.exists(CLIP_PATH) and os.path.getsize(CLIP_PATH) > 100_000):
    if not (os.path.exists(PBF_PATH) and os.path.getsize(PBF_PATH) > 1_000_000):
        print("downloading", EXTRACT_URL)
        if subprocess.call(["wget", "-q", "--show-progress", "-O", PBF_PATH, EXTRACT_URL]) != 0:
            raise SystemExit("Download failed — check EXTRACT_URL.")
    x0, y0, x1, y1 = STUDY_BBOX
    b = CLIP_BUFFER_DEG
    bbox = f"{x0-b},{y0-b},{x1+b},{y1+b}"
    print("clipping to buffered box", bbox)
    r = subprocess.run(["osmium", "extract", "-b", bbox, PBF_PATH, "-o", CLIP_PATH,
                        "--overwrite", "-s", "complete_ways"], capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stderr[-2000:]); raise SystemExit("Clip failed.")

EXTRACT_DATE = pairing.extract_date(PBF_PATH)
print(round(os.path.getsize(CLIP_PATH)/1e6, 1), "MB clipped | extract dated", EXTRACT_DATE)
if EXTRACT_DATE is None:
    print("WARNING: the extract header carries no date. Records will say so (null), not guess.")

## 4. Parse, build the graph, pair, clip, route

In [ ]:
import time
t0 = time.time()
g = pairing.load_network(CLIP_PATH)
graph = pairing.build_graph(g.ways, g.coords)
run = pairing.run_pairing(g.ways, g.coords, config=CONFIG, study_bbox=STUDY_BBOX,
                          osm_extract_date=EXTRACT_DATE, sha=SHA, graph=graph)
print(f"done in {time.time()-t0:.0f}s | military-tagged ways stripped: {g.military_stripped}")

import pandas as pd
display(pd.Series({k: v for k, v in run.meta.items() if k != "config"}, name="run").to_frame())

### Rejections by zone (F-8)

In [ ]:
rej = pd.DataFrame([{"zone": z, "reason": r, "count": c} for (z, r), c in run.rejections.items()])
if not rej.empty:
    display(rej.pivot_table(index="reason", columns="zone", values="count", aggfunc="sum", fill_value=0))

## 5. Headline — every routed point, nothing removed (F-6)

Pairs whose endpoints sit in different graph components have no route in our data. That is a gap, not a detour, so they are reported separately.

In [ ]:
res = pd.DataFrame(run.records)
routed = res[res.detour_index.notna() & ~res.diff_component]
print(f"pairs kept {len(res):,} | routed {len(routed):,} | "
      f"null routes {dict(run.meta['null_routes'])} | near limit {int(res.near_limit.sum()):,}")
h = metrics.headline_stats(routed.detour_index.tolist())
display(pd.Series(h, name="all routed, uncensored").to_frame())
print(f"Report the uncensored median {h['median']:.2f}, never the censored {h['censored_median']:.2f}.")

## 6. Export — validated, or not written at all

In [ ]:
import datetime, shutil
stamp = datetime.date.today().isoformat()
tag = (SHA or "nosha")[:7]
points_path = f"{OUT_DIR}/di_points_{stamp}_{tag}.csv"

pairing.write_point_file(run.records, points_path, run.meta)   # raises on any invalid record
print("validated and written:", points_path)

drive.mount("/content/drive")           # no-op if still mounted from cell 2
os.makedirs(DRIVE_DEST, exist_ok=True)  # only the run folder; DRIVE_BASE was checked in cell 2
for fn in os.listdir(OUT_DIR):
    shutil.copy(f"{OUT_DIR}/{fn}", f"{DRIVE_DEST}/{fn}")
print("saved to Drive:", DRIVE_DEST)

## What these numbers are and are not

- Distance and geometry, for **driving**. Never walking, never travel time.
- Candidates. No absolute Detour Index value is published until a modelled route
  has been checked against a driven one.
- Map data © OpenStreetMap contributors (ODbL). Measurements by arcroad (ODbL).